In [1]:
pip install pyserial requests

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\Benize\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import serial
import requests
import time
import json

# =====================================================
# SETTINGS
# =====================================================

COM_PORT = "COM5"
BAUD_RATE = 9600

LM_STUDIO_URL = "http://127.0.0.1:1234/v1/chat/completions"

# =====================================================
# CONNECT TO ARDUINO
# =====================================================

try:

    arduino = serial.Serial(
        COM_PORT,
        BAUD_RATE,
        timeout=1
    )

    time.sleep(3)

    print("Arduino Connected.")

except Exception as e:

    print("Arduino Error:", e)

    arduino = None

# =====================================================
# CHAT HISTORY
# =====================================================

chat_history = []

# =====================================================
# READ SENSOR DATA
# =====================================================

def read_sensor_data():

    default_data = "Gas: 0 Temp: 0 Tilt: 0"

    if arduino is None:
        return default_data

    try:

        # CLEAR BUFFER
        arduino.reset_input_buffer()

        # READ LINE
        sensor_data = arduino.readline().decode(
            "utf-8",
            errors="ignore"
        ).strip()

        # EMPTY CHECK
        if sensor_data == "":
            return default_data

        return sensor_data

    except Exception as e:

        print("Sensor Error:", e)

        return default_data

# =====================================================
# PARSE SENSOR DATA
# =====================================================

def parse_sensor_data(sensor_data):

    result = {
        "gas": 0,
        "temp": 0,
        "tilt": 0
    }

    try:

        tokens = sensor_data.split()

        result["gas"] = int(tokens[1])
        result["temp"] = float(tokens[3])
        result["tilt"] = int(tokens[5])

    except:
        pass

    return result

# =====================================================
# SAFETY ANALYSIS
# =====================================================

def analyze_safety(values):

    warnings = []

    gas = values["gas"]
    temp = values["temp"]
    tilt = values["tilt"]

    if gas >= 400:
        warnings.append("CRITICAL GAS LEAK")

    elif gas >= 250:
        warnings.append("HIGH GAS LEVEL")

    if temp >= 60:
        warnings.append("EXTREME HEAT")

    elif temp >= 45:
        warnings.append("HIGH TEMPERATURE")

    if tilt == 1:
        warnings.append("GAS TANK UNSTABLE")

    if len(warnings) == 0:
        warnings.append("Kitchen conditions are safe.")

    return warnings

# =====================================================
# ASK AI
# =====================================================

def ask_ai(food_input, sensor_data, warnings):

    payload = {

        "messages": [

            {
                "role": "system",
                "content": """
You are an AI Kitchen Assistant.

Tasks:
- Estimate calories
- Give short cooking advice
- Analyze kitchen safety

Keep answers short.
Maximum 40 words.
"""
            },

            {
                "role": "user",
                "content": f"""
Food: {food_input}

Sensors:
{sensor_data}

Warnings:
{warnings}
"""
            }

        ],

        "temperature": 0.6,
        "max_tokens": 80

    }

    try:

        response = requests.post(
            LM_STUDIO_URL,
            json=payload,
            timeout=30
        )

        data = response.json()

        if "choices" in data:

            return data["choices"][0]["message"]["content"]

        elif "error" in data:

            return f"LM Studio Error: {data['error']}"

        else:

            return "Invalid AI response."

    except Exception as e:

        return f"AI Error: {e}"

# =====================================================
# SEND TO LCD
# =====================================================

def send_to_lcd(message):

    if arduino is None:
        return

    try:

        message = message.replace("\n", " ")

        message = message[:32]

        line1 = message[:16]
        line2 = message[16:32]

        arduino.write((line1 + "\n").encode())
        time.sleep(0.5)

        arduino.write((line2 + "\n").encode())
        time.sleep(0.5)

    except Exception as e:

        print("LCD Error:", e)

# =====================================================
# SAVE HISTORY
# =====================================================

def save_history():

    try:

        with open("kitchen_history.json", "w") as file:

            json.dump(chat_history, file, indent=4)

    except Exception as e:

        print("Save Error:", e)

# =====================================================
# MAIN LOOP
# =====================================================

print("=" * 60)
print("AI KITCHEN GUARDIAN STARTED")
print("=" * 60)

while True:

    # =================================================
    # READ SENSOR DATA
    # =================================================

    sensor_data = read_sensor_data()

    print("\nSensor Data:", sensor_data)

    # =================================================
    # PARSE VALUES
    # =================================================

    sensor_values = parse_sensor_data(sensor_data)

    # =================================================
    # SAFETY ANALYSIS
    # =================================================

    warnings = analyze_safety(sensor_values)

    print("\nSafety Analysis:")

    for warning in warnings:
        print("-", warning)

    # =================================================
    # USER INPUT
    # =================================================

    food_input = input(
        "\nEnter food (example: chicken 200g)\nType QUIT to exit:\n> "
    ).strip()

    # EMPTY INPUT FIX
    if food_input == "":
        continue

    # EXIT
    if food_input.upper() == "QUIT":
        break

    # =================================================
    # ASK AI
    # =================================================

    print("\nThinking...\n")

    ai_response = ask_ai(
        food_input,
        sensor_data,
        warnings
    )

    # =================================================
    # DISPLAY
    # =================================================

    print("=" * 60)

    print("USER:")
    print(food_input)

    print("\nAI RESPONSE:")
    print(ai_response)

    print("=" * 60)

    # =================================================
    # SAVE HISTORY
    # =================================================

    chat_history.append({

        "user": food_input,
        "sensor_data": sensor_data,
        "warnings": warnings,
        "ai_response": ai_response

    })

    save_history()

    # =================================================
    # SEND LCD
    # =================================================

    send_to_lcd(ai_response)

# =====================================================
# CLEANUP
# =====================================================

if arduino:
    arduino.close()

print("\nAI Kitchen Guardian Stopped.")

Arduino Connected.
AI KITCHEN GUARDIAN STARTED

Sensor Data: Gas: 96 Temp: 30.10 Tilt: 0

Safety Analysis:
- Kitchen conditions are safe.

Thinking...

USER:
Hello!

AI RESPONSE:
Hello! Safe kitchen conditions detected.

Sensor Data: Gas: 95 Temp: 30.10 Tilt: 0

Safety Analysis:
- Kitchen conditions are safe.

Thinking...

USER:
Can you give me the count of calories of Chicken 200g

AI RESPONSE:
Calories: Approximately 300 kcal

Short cooking advice: Season chicken with salt and pepper, then bake until golden brown on both sides.

Kitchen safety: Keep the oven door closed during cooking to prevent fire hazards.

Sensor Data: Gas: 90 Temp: 30.10 Tilt: 0

Safety Analysis:
- Kitchen conditions are safe.

Thinking...

USER:
how about rice 150g

AI RESPONSE:
Rice contains around 120 calories per 150g. For short cooking advice, consider steaming or boiling rice without adding water to avoid overcooking. Safe kitchen conditions are assured by the provided information.

Sensor Data: Gas: 85 Te